In [ ]:
"""#**2_feature_engineering**"""



"""##**Load Data**"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

df = spark.read.parquet("data/amazon_reviews_parquet")
df.show(5)

"""##**Select Text + Rating**"""

df = df.select(
    col("text").alias("review_text"),
    col("rating").alias("stars")
)

df = df.withColumn("label", col("stars") - 1)

"""##**Split Train/Test**"""

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

"""##**NLP Pipeline**"""

from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

tokenizer = Tokenizer(inputCol="review_text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=20000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf])

model = pipeline.fit(train_df)

train_data = model.transform(train_df)
test_data = model.transform(test_df)

"""##**Persist Features**"""

train_data.persist()
test_data.persist()

train_data.count()

"""**Save Feature Data**"""

train_data.write.mode("overwrite").parquet("data/train_features")
test_data.write.mode("overwrite").parquet("data/test_features")